# 13 — AI benchmark and evidence comparison

**Project:** Hotel overbooking decision support  
**Stage:** Review AI-assisted work and benchmark evidence  
**Purpose:** Separate what our analytical experiments establish from what the assignment's independent AI build still needs to prove.

This notebook has two distinct parts:

1. **Existing analytical benchmark:** inspect the team's saved model-family, calibration and policy-comparison results under `evidence/expanded_benchmark/`.
2. **Assignment AI build audit:** check whether the separate `ai-benchmark` branch artifacts (`PLAN_AI.md`, `run.py`, and `RESULT.md`) are actually available for comparison.

These are not interchangeable. A large internal model experiment is not the same thing as the assignment's independent vendor-style AI build.

**Honesty rule:** If the AI branch and its run evidence are not present, this notebook records the gap. It must not invent AI outputs, mistakes, or a line-by-line comparison.

In [1]:
# Run from the repository root, so `from src import ...` and the relative evidence/ paths work
# whether this notebook is opened from the repo folder or from notebooks/pipeline/.
import os
import sys
from pathlib import Path

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "contract.py").exists())
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print("Working directory:", REPO_ROOT)

Working directory: C:\Users\Lenovo\Desktop\Build_2_Repo\IIT_GN_W06


## 1. Read the benchmark protocol

The assignment's `docs/AI_WORKFLOW.md` specifies that the AI benchmark should:
- start from the repository's first commit on a separate `ai-benchmark` branch;
- give the AI only `BRIEF.md`, `docs/EVALUATION_CONTRACT.md`, `src/data.py`, and `src/contract.py`;
- ask for a plan only, then save and approve that plan;
- ask the AI to implement `ai_benchmark/run.py`;
- run the script end to end and save its output as `RESULT.md`;
- avoid editing AI logic except to fix an environment error;
- keep the branch separate and never merge it into `main`.

The required execution window is 90 minutes. The chat transcript belongs in `ai_logs/`, with a row in `ai_logs/INDEX.md`.

This notebook does not recreate that experiment after the fact. It audits whether the artifacts exist and analyses the evidence already saved.

## 2. Load the existing expanded benchmark outputs

The existing experiment files are expected at:
- `evidence/expanded_benchmark/cv_leaderboard.csv`
- `evidence/expanded_benchmark/calibration_comparison.csv`
- `evidence/expanded_benchmark/choose_policy_comparison.csv`
- `evidence/expanded_benchmark/proof_audit_all_models.csv`
- `evidence/expanded_benchmark/run_metadata.json`

If a file is absent, investigate the repository layout rather than inventing a replacement result.

In [2]:
import json
import pandas as pd
from pathlib import Path
from IPython.display import display

evidence_folder = Path("evidence/expanded_benchmark")

required_files = [
    "cv_leaderboard.csv",
    "calibration_comparison.csv",
    "choose_policy_comparison.csv",
    "proof_audit_all_models.csv",
    "run_metadata.json"
]

file_status = pd.DataFrame([
    {
        "file": filename,
        "exists": (evidence_folder / filename).exists()
    }
    for filename in required_files
])

display(file_status)

missing_files = file_status.loc[~file_status["exists"], "file"].tolist()

if missing_files:
    print("Missing benchmark evidence files:", missing_files)
    print("Do not proceed with conclusions that depend on missing files.")
else:
    cv_leaderboard = pd.read_csv(evidence_folder / "cv_leaderboard.csv")
    calibration_comparison = pd.read_csv(evidence_folder / "calibration_comparison.csv")
    choose_policy_comparison = pd.read_csv(evidence_folder / "choose_policy_comparison.csv")
    proof_audit = pd.read_csv(evidence_folder / "proof_audit_all_models.csv")

    with open(evidence_folder / "run_metadata.json", encoding="utf-8") as file:
        run_metadata = json.load(file)

    print("Loaded the saved expanded benchmark evidence.")

,file,exists
0,cv_leaderboard.csv,True
1,calibration_comparison.csv,True
2,choose_policy_comparison.csv,True
3,proof_audit_all_models.csv,True
4,run_metadata.json,True


Loaded the saved expanded benchmark evidence.


## 3. Inspect the experiment metadata

The metadata records the dataset fingerprint, software versions, time split, feature list and experiment counts. These details matter when judging reproducibility.

The existing benchmark metadata reports:
- 119,390 rows;
- target `is_canceled`;
- 3-fold expanding-window time-series validation;
- Train period ending 30 June 2016;
- a known risk that the source snapshot is not proven to be an original booking-time snapshot.

The exact run metadata loaded from the repository is the authority if it differs from this summary.

In [3]:
if not missing_files:
    metadata_summary = {
        "run_id": run_metadata.get("run_id"),
        "created_utc": run_metadata.get("created_utc"),
        "dataset_sha256": run_metadata.get("dataset_sha256"),
        "rows": run_metadata.get("rows"),
        "target": run_metadata.get("target"),
        "features": ", ".join(run_metadata.get("features", [])),
        "cv": run_metadata.get("cv"),
        "train_period": str(run_metadata.get("train_period")),
        "known_risk": run_metadata.get("known_risk"),
        "experiment_counts": str(run_metadata.get("experiment_counts"))
    }

    display(pd.DataFrame(
        [{"field": key, "value": value} for key, value in metadata_summary.items()]
    ))

,field,value
0,run_id,hotel-cancel-20261115-061831
1,created_utc,2026-11-15T06:18:31.200504+00:00
2,dataset_sha256,d10cdeb07de9309007f528d6499a2c0467ae7a6b287b80...
3,rows,119390
4,target,is_canceled
5,features,"hotel, country, market_segment, distribution_c..."
6,cv,3-fold expanding-window TimeSeriesSplit on arr...
7,train_period,"['2015-07-01', '2016-06-30']"
8,known_risk,source snapshot not proven to be original book...
9,experiment_counts,"{'cv_model_and_baseline_configurations': 27, '..."


## 4. Compare model families on chronological validation

The leaderboard contains different model families and simple baselines. Brier score measures probability accuracy; lower is better. ROC AUC measures ranking; higher is better.

Do not choose a model by ROC AUC alone if the operating decision requires reliable probabilities.

In [4]:
if not missing_files:
    completed_cv = cv_leaderboard[
        cv_leaderboard["status"] == "completed"
    ].copy()

    cv_display_columns = [
        "model",
        "folds_completed",
        "cv_auc_mean",
        "cv_brier_mean",
        "cv_logloss_mean",
        "runtime_seconds"
    ]

    display(
        completed_cv[cv_display_columns]
        .sort_values("cv_brier_mean")
        .reset_index(drop=True)
    )

    best_brier_row = completed_cv.sort_values(
        "cv_brier_mean"
    ).iloc[0]

    best_auc_row = completed_cv.sort_values(
        "cv_auc_mean",
        ascending=False
    ).iloc[0]

    print("Lowest mean CV Brier score:", best_brier_row["model"])
    print("Highest mean CV ROC AUC:", best_auc_row["model"])

,model,folds_completed,cv_auc_mean,cv_brier_mean,cv_logloss_mean,runtime_seconds
0,Random forest balanced,3,0.835390,0.147312,0.456372,3.340000
1,Random forest,3,0.836662,0.150616,0.466853,2.650000
2,Gradient boosting,3,0.827212,0.150940,0.467138,6.210000
3,SGD log-loss,3,0.831160,0.152738,0.512360,0.500000
4,Logistic + undersampling,3,0.831305,0.152757,0.488839,0.660000
5,XGBoost,3,0.838503,0.152911,0.501162,1.020000
6,Decision tree,3,0.843851,0.153097,0.503665,0.440000
7,Logistic + random oversampling,3,0.827994,0.154087,0.496406,1.030000
8,Logistic + SMOTE,3,0.828053,0.154109,0.499130,5.640000
9,LightGBM,3,0.842544,0.154164,0.512133,1.120000


Lowest mean CV Brier score: Random forest balanced
Highest mean CV ROC AUC: Decision tree


## 5. Inspect calibration results

Calibration methods are compared using out-of-fold Brier scores before and after calibration. The benchmark's reported table can show whether calibration improved those out-of-fold scores.

This is evidence from the saved experiment. It does not guarantee that calibration will remain effective under future distribution shift.

In [5]:
if not missing_files:
    calibration_comparison["brier_change"] = (
        calibration_comparison["oof_brier_after"]
        - calibration_comparison["oof_brier_before"]
    )

    calibration_comparison["improved"] = (
        calibration_comparison["brier_change"] < 0
    )

    display(
        calibration_comparison.sort_values(
            "oof_brier_after"
        ).reset_index(drop=True)
    )

    print(
        "Calibration configurations with lower OOF Brier:",
        int(calibration_comparison["improved"].sum()),
        "of",
        len(calibration_comparison)
    )

,model,calibration,oof_n,oof_brier_before,oof_brier_after,brier_change,improved
0,Random forest balanced,isotonic,37257,0.149736,0.142063,-0.007673,True
1,XGBoost,isotonic,37257,0.157186,0.145689,-0.011497,True
2,LightGBM,isotonic,37257,0.159203,0.145847,-0.013355,True
3,MLP small,isotonic,37257,0.160017,0.145910,-0.014107,True
4,Random forest,isotonic,37257,0.155246,0.146413,-0.008833,True
5,Random forest balanced,sigmoid,37257,0.149736,0.149087,-0.000650,True
6,LightGBM,sigmoid,37257,0.159203,0.150418,-0.008784,True
7,XGBoost,sigmoid,37257,0.157186,0.150536,-0.006650,True
8,Random forest,sigmoid,37257,0.155246,0.151720,-0.003526,True
9,Logistic elastic-net,isotonic,37257,0.162424,0.152304,-0.010120,True


Calibration configurations with lower OOF Brier: 12 of 12


## 6. Compare Choose-period operating policies

The Choose comparison reports cost, simulated empty rooms, simulated walks and walks per 1,000 arrivals.

The formal selection rule is:
1. retain policies that meet the client's walk ceiling;
2. among those, compare total contract cost.

A policy with lower cost but a failed walk ceiling is not acceptable.

In [6]:
if not missing_files:
    acceptable_choose = choose_policy_comparison[
        choose_policy_comparison["ceiling_met"] == True
    ].copy()

    display(
        choose_policy_comparison.sort_values(
            ["ceiling_met", "cost_eur"],
            ascending=[False, True]
        ).reset_index(drop=True)
    )

    if acceptable_choose.empty:
        print("No saved Choose policy meets the walk ceiling.")
    else:
        best_choose_policy = acceptable_choose.sort_values(
            "cost_eur"
        ).iloc[0]

        print("Lowest-cost saved policy meeting the ceiling:")
        display(best_choose_policy.to_frame("value"))

,method,policy,parameter,cost_eur,empty_rooms,walked_guests,walks_per_1000,ceiling_met,nights
0,Random forest + isotonic,uncertainty,1.25,1295631.50,2921,180,9.65,True,368
1,MLP small + isotonic,uncertainty,1.80,1305355.13,3142,184,9.86,True,368
2,Random forest balanced + isotonic,uncertainty,1.65,1305726.18,2950,186,9.97,True,368
3,MLP small,uncertainty,0.25,1308468.96,3115,184,9.86,True,368
4,XGBoost + isotonic,uncertainty,1.70,1337377.44,3108,182,9.76,True,368
5,LightGBM + isotonic,uncertainty,1.95,1339077.58,3186,180,9.65,True,368
6,LightGBM + sigmoid,uncertainty,2.00,1357628.95,3193,184,9.86,True,368
7,XGBoost + sigmoid,uncertainty,1.95,1367363.67,3193,177,9.49,True,368
8,Random forest balanced + sigmoid,uncertainty,1.90,1371116.97,3115,182,9.76,True,368
9,Random forest + sigmoid,uncertainty,1.60,1389576.17,3171,185,9.92,True,368


Lowest-cost saved policy meeting the ceiling:


,value
method,Random forest + isotonic
policy,uncertainty
parameter,1.25
cost_eur,1295631.5
empty_rooms,2921
walked_guests,180
walks_per_1000,9.65
ceiling_met,True
nights,368


## 7. Treat the Proof audit as exposed historical evidence

The saved Proof audit labels its rows `AUDIT_ONLY_PROOF_ALREADY_EXPOSED`. That label matters.

These values are useful for understanding how selected Choose policies behaved on the historically inspected Proof period. They are not a clean, untouched final test and must not be used to retune the policy.

The table can also reveal that a policy which met the Choose guardrail failed it on Proof. This is a reason to discuss temporal variation and operational risk, not a reason to change the frozen policy after the fact.

In [7]:
if not missing_files:
    proof_display_columns = [
        "method",
        "choose_policy",
        "choose_cost_eur",
        "choose_walks_per_1000",
        "cost_eur",
        "walked_guests",
        "walks_per_1000",
        "ceiling_met",
        "proof_status"
    ]

    display(
        proof_audit[proof_display_columns]
        .sort_values("cost_eur")
        .reset_index(drop=True)
    )

    print("Proof rows marked audit-only:",
          proof_audit["proof_status"].astype(str).str.contains(
              "PROOF_ALREADY_EXPOSED"
          ).sum())

,method,choose_policy,choose_cost_eur,choose_walks_per_1000,cost_eur,walked_guests,walks_per_1000,ceiling_met,proof_status
0,MLP small,uncertainty,1308468.96,9.86,2521115.62,291,11.67,False,AUDIT_ONLY_PROOF_ALREADY_EXPOSED
1,LightGBM + isotonic,uncertainty,1339077.58,9.65,2542346.17,236,9.46,True,AUDIT_ONLY_PROOF_ALREADY_EXPOSED
2,XGBoost + isotonic,uncertainty,1337377.44,9.76,2545459.73,237,9.50,True,AUDIT_ONLY_PROOF_ALREADY_EXPOSED
3,MLP small + sigmoid,uncertainty,1393469.48,9.54,2546214.06,249,9.98,True,AUDIT_ONLY_PROOF_ALREADY_EXPOSED
4,MLP small + isotonic,uncertainty,1305355.13,9.86,2546239.73,272,10.91,False,AUDIT_ONLY_PROOF_ALREADY_EXPOSED
5,Random forest + isotonic,uncertainty,1295631.50,9.65,2549978.60,260,10.42,False,AUDIT_ONLY_PROOF_ALREADY_EXPOSED
6,LightGBM + sigmoid,uncertainty,1357628.95,9.86,2556073.57,235,9.42,True,AUDIT_ONLY_PROOF_ALREADY_EXPOSED
7,XGBoost + sigmoid,uncertainty,1367363.67,9.49,2584628.31,230,9.22,True,AUDIT_ONLY_PROOF_ALREADY_EXPOSED
8,Random forest balanced + isotonic,uncertainty,1305726.18,9.97,2622383.62,264,10.58,False,AUDIT_ONLY_PROOF_ALREADY_EXPOSED
9,Random forest balanced + sigmoid,uncertainty,1371116.97,9.76,2641679.68,243,9.74,True,AUDIT_ONLY_PROOF_ALREADY_EXPOSED


Proof rows marked audit-only: 22


## 8. Check whether the assignment's separate AI branch artifacts exist

The assignment benchmark expects these artifacts on the separate `ai-benchmark` branch:
- `ai_benchmark/PLAN_AI.md`
- `ai_benchmark/run.py`
- `ai_benchmark/RESULT.md`

This notebook checks the working tree paths. If the branch was not checked out or its files were not copied for inspection, the result will show that the branch-specific comparison is still outstanding.

A missing file is not evidence that the AI failed. It means the evidence is unavailable here.

In [8]:
ai_artifact_paths = [
    Path("ai_benchmark/PLAN_AI.md"),
    Path("ai_benchmark/run.py"),
    Path("ai_benchmark/RESULT.md")
]

ai_artifact_status = pd.DataFrame([
    {
        "artifact": str(path),
        "exists_in_current_working_tree": path.exists(),
        "bytes": path.stat().st_size if path.exists() else None
    }
    for path in ai_artifact_paths
])

display(ai_artifact_status)

ai_branch_artifacts_available = ai_artifact_status[
    "exists_in_current_working_tree"
].all()

print("All three AI branch artifacts available:", ai_branch_artifacts_available)

,artifact,exists_in_current_working_tree,bytes
0,ai_benchmark\PLAN_AI.md,False,None
1,ai_benchmark\run.py,False,None
2,ai_benchmark\RESULT.md,False,None


All three AI branch artifacts available: False


## 9. Record the actual AI build findings only after inspecting its branch

If the branch is available, compare its actual plan, code and output against:
- `BRIEF.md`;
- `docs/EVALUATION_CONTRACT.md`;
- `src/data.py`;
- `src/contract.py`;
- the team's implementation and documented decisions.

For each finding, record:
- exact AI branch file and line or section;
- what the AI did;
- the requirement or contract clause involved;
- why it is a gap or mistake;
- evidence supporting the finding;
- consequence for the recommendation;
- whether it is a confirmed issue or only a concern.

Do not infer an AI mistake from a difference in modelling choice alone. A different choice can be reasonable if it follows the brief and contract.

In [9]:
# Fill these rows only after reviewing the actual AI branch.
# Leave the table empty rather than inventing findings.
ai_branch_findings = pd.DataFrame(columns=[
    "finding_id",
    "ai_branch_file_and_line",
    "ai_implementation_or_claim",
    "requirement_or_contract_clause",
    "evidence",
    "gap_or_mistake",
    "consequence",
    "status"
])

display(ai_branch_findings)
print("Findings recorded:", len(ai_branch_findings))

,finding_id,ai_branch_file_and_line,ai_implementation_or_claim,requirement_or_contract_clause,evidence,gap_or_mistake,consequence,status


Findings recorded: 0


## 10. Current comparison status

The repository's existing comparison document states that the separate assignment AI branch's code, commit history and run logs have not been supplied as an inspectable branch in the working repository.

It also identifies three gaps in the team's own build:
1. No booking-time snapshot contract.
2. The nightly CLI does not filter input to an as-of active booking population.
3. The contract uses booking-equivalents, not room inventory or observed walks.

These are team-build gaps. They are not proven mistakes in the separate AI build.

In [10]:
current_status = pd.DataFrame([
    {
        "comparison_area": "Expanded internal model benchmark",
        "status": "Evidence files available in evidence/expanded_benchmark",
        "what_can_be_claimed": "The saved experiment metrics can be analysed and reported"
    },
    {
        "comparison_area": "Separate assignment AI branch",
        "status": (
            "Artifacts available for inspection"
            if ai_branch_artifacts_available
            else "Not available in current working tree"
        ),
        "what_can_be_claimed": (
            "Branch-specific comparison can proceed after reviewing actual files"
            if ai_branch_artifacts_available
            else "No line-by-line AI comparison or AI mistake list can yet be claimed"
        )
    },
    {
        "comparison_area": "Team implementation gaps",
        "status": "Three gaps documented in docs/COMPARISON.md",
        "what_can_be_claimed": "These are documented team-build limitations, not AI branch findings"
    }
])

display(current_status)

,comparison_area,status,what_can_be_claimed
0,Expanded internal model benchmark,Evidence files available in evidence/expanded_...,The saved experiment metrics can be analysed a...
1,Separate assignment AI branch,Not available in current working tree,No line-by-line AI comparison or AI mistake li...
2,Team implementation gaps,Three gaps documented in docs/COMPARISON.md,"These are documented team-build limitations, n..."


## 11. Save this notebook's benchmark review outputs

The notebook writes a small review summary under `evidence/ai_benchmark_review/`. It does not fabricate or recreate the missing AI branch artifacts.

In [11]:
review_folder = Path("evidence/ai_benchmark_review")
review_folder.mkdir(parents=True, exist_ok=True)

current_status.to_csv(
    review_folder / "comparison_status.csv",
    index=False
)

ai_artifact_status.to_csv(
    review_folder / "ai_branch_artifact_status.csv",
    index=False
)

ai_branch_findings.to_csv(
    review_folder / "ai_branch_findings.csv",
    index=False
)

if not missing_files:
    completed_cv.to_csv(
        review_folder / "internal_cv_leaderboard_review.csv",
        index=False
    )
    calibration_comparison.to_csv(
        review_folder / "internal_calibration_review.csv",
        index=False
    )
    choose_policy_comparison.to_csv(
        review_folder / "internal_choose_policy_review.csv",
        index=False
    )
    proof_audit.to_csv(
        review_folder / "internal_proof_audit_review.csv",
        index=False
    )

print("Saved AI benchmark review evidence to:", review_folder)

Saved AI benchmark review evidence to: evidence\ai_benchmark_review


## 12. Questions for your project defence

1. Why is the expanded internal model benchmark not the same as the assignment's independent AI build?
2. Why must the AI receive only the four files named in the assignment workflow?
3. Why is the AI asked to produce a plan before writing code?
4. Why must the AI branch remain separate from `main`?
5. What is the difference between a confirmed AI mistake and a concern?
6. Why can the team's documented gaps not be presented as AI mistakes?
7. Why does the `AUDIT_ONLY_PROOF_ALREADY_EXPOSED` label matter?

### Final status

Until the separate AI branch's `PLAN_AI.md`, `run.py`, `RESULT.md`, commit history and logs have been inspected, the branch-specific comparison remains incomplete. Keep that statement visible in `docs/COMPARISON.md`.

**Next notebook:** `14_nightly_pipeline_validation.ipynb`, where we inspect the nightly inference prototype and test what it can and cannot support operationally.